Functions to be tested. 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Point

from geopandas import GeoDataFrame
import zipfile

import sys
import os
sys.path.append("../..")
import movingpandas as mpd

import warnings
import numpy as np
warnings.simplefilter("ignore")
from pyproj import Transformer
from collections import Counter
from itertools import compress

import pymeos
from pymeos.mixins.simplify import TSimplifiable
from pymeos import TGeomPointSeq
import time
from memory_profiler import memory_usage

pymeos.pymeos_initialize()

In [ ]:
zip_path = ['data/aisdk-2025-07-11.zip', 'data/aisdk-2025-07-12.zip']
csv_filename = ['aisdk-2025-07-11.csv', 'aisdk-2025-07-12.csv']

sizes = {'small': 100000, 'medium': 1000000, 'large': 4000000}

list_of_selected_mmsi = [2579999, 219236000, 2190049, 2190068, 2190048, 2190069, 257076860, 258092000, 2190067, 235108534, 
                         2655148, 259490000, 265859000, 2190071, 219022903, 2190051, 2190073, 219007781, 247389200, 3638]

In [ ]:
def extraction(chosen_size):
    with zipfile.ZipFile(zip_path[0]) as archive:        
        with archive.open(csv_filename[0]) as file:
            df = pd.read_csv(file, usecols=['# Timestamp', 'MMSI', 'Latitude', 'Longitude'], nrows=sizes[chosen_size])

    df.rename(columns={'# Timestamp': 't', 'MMSI': 'mmsi', 'Latitude': 'lat', 'Longitude': 'lon'}, inplace=True)
    df = df[df['lat'] <= 90] # default latitude = 91°
    df = df.sort_values(['mmsi', 't'])
    df = df.drop_duplicates(subset=['mmsi', 't'], keep='last')

    transformer = Transformer.from_crs("EPSG:4326", "EPSG:25832", always_xy=True)

    df[['x', 'y']] = df.apply(
        lambda row: pd.Series(transformer.transform(row['lon'], row['lat'])),
        axis=1
    )
    df = df.drop(columns=['lat', 'lon'])

    return df

In [ ]:
def construction_TC(df):
    df1 = df.copy()
    df1['geometry'] = [Point(x, y) for x, y in zip(df1['x'], df1['y'])]
    gdf = GeoDataFrame(df1, geometry='geometry')
    gdf.set_crs(epsg=25832, inplace=True)
    gdf['t'] = pd.to_datetime(gdf['t'])

    return mpd.TrajectoryCollection(gdf, traj_id_col='mmsi', t='t')

In [ ]:
def TC_to_TG(trajectories_without):

    df_all = pd.concat([traj.df for traj in trajectories_without.trajectories], ignore_index=False)

    tr = (df_all.groupby('mmsi')
        .apply(lambda traj: TGeomPointSeq.from_arrays(
            t = traj.index.strftime('%Y-%m-%d %H:%M:%S').values,
            x = traj['x'].values, 
            y = traj['y'].values,
            srid = 25832,
            upper_inc=True,
        ))
        .rename('trajectory')
    ).to_frame()

    return tr['trajectory'].apply(
        lambda seq: TGeomPointSeq.from_instants(seq.instants(), upper_inc=True)
    ).to_frame(name='trajectory')

In [ ]:
def simplify_pymeos(trajectories_with):
    return trajectories_with.trajectory.apply(
        lambda x: TSimplifiable.simplify_douglas_peucker(x, distance=1, synchronized=False)
    )

In [ ]:
def TG_to_TC(simplified_with): # naive

    trajs = []
    for idx, row in simplified_with.to_frame().iterrows():
        seq = row['trajectory']

        # get the instants
        times, points = [], []
        for inst in seq.instants():
            st, end = inst.start_timestamp(), inst.end_timestamp()
            times.append(st)
            if end and end != st:
                times.append(end)
            points.append(inst.value())

        gdf = GeoDataFrame(geometry=points, index=pd.to_datetime(times))
        gdf.index.name = 't'

        traj = mpd.Trajectory(gdf, traj_id=idx)
        trajs.append(traj)

    return mpd.TrajectoryCollection(trajs)

In [ ]:
def TG_to_TC_fast(simplified_with): 

    ids, times, geoms = [], [], []

    for idx, traj in simplified_with.items():

        insts = traj.instants()
        starts = [inst.start_timestamp() for inst in insts]
        ends = [inst.end_timestamp() for inst in insts]
        vals = [inst.value() for inst in insts]

        mask = [e != s for s, e in zip(starts, ends)]
        extra_times = list(compress(ends, mask))
        extra_vals  = list(compress(vals,  mask))

        times_part = starts + extra_times
        geoms_part = vals   + extra_vals
        ids_part = [idx] * (len(times_part))

        ids += ids_part
        times += times_part
        geoms += geoms_part
        

    df_all = pd.DataFrame({'mmsi': ids, 'geometry': geoms}, index=pd.to_datetime(times))
    df_all.index.name = 't'
    gdf_all = GeoDataFrame(df_all, geometry='geometry')

    trajs = []
    for mmsi, traj in gdf_all.groupby('mmsi', sort=False):
        traj = traj.drop(columns='mmsi')
        trajs.append(mpd.Trajectory(traj, traj_id=mmsi))

    return mpd.TrajectoryCollection(trajs)